# 类继承关系
```mermaid
classDiagram
    class MetaStatsBuilderMixin {
        <<metaclass>>
    }
    class MetaPlotsBuilderMixin {
        <<metaclass>>
    }
    class StatsBuilderMixin
    class PlotsBuilderMixin
    class MetaPortfolio {
        <<metaclass>>
    }

    class Wrapping
    class Portfolio

    
    %% 继承关系
    MetaStatsBuilderMixin <|-- StatsBuilderMixin : metaclass
    MetaPlotsBuilderMixin <|-- PlotsBuilderMixin : metaclass

    MetaStatsBuilderMixin <|-- MetaPortfolio : metaclass
    MetaPlotsBuilderMixin <|-- MetaPortfolio : metaclass

    Wrapping <|-- Portfolio
    StatsBuilderMixin <|-- Portfolio
    PlotsBuilderMixin <|-- Portfolio
    MetaPortfolio <|-- Portfolio : metaclass
```

# class Portfolio(Wrapping, StatsBuilderMixin, PlotsBuilderMixin, metaclass=MetaPortfolio)

## `__init__` 
参数
- `wrapper (ArrayWrapper)`：数组包装器，负责管理数据的索引、形状、分组等操作。
- `close (array_like)`：每个时间步的最后资产价格
- `order_records (array_like)`：订单记录的结构化NumPy数组
  - 包含所有已执行订单的详细信息，如价格、数量、费用等。
- `log_records (array_like)`：日志记录的结构化NumPy数组
- `init_cash (InitCashMode, float or array_like of float)`：初始资本
- `cash_sharing (bool)`：是否在同一组内共享现金
- `call_seq (array_like of int)`：每行每组的调用序列，默认为None
  - 控制订单在同一时间步内的执行顺序。
- `fillna_close (bool)`：是否前向和后向填充close中的NaN值
- `trades_type (str or int)`：默认的交易类型

```python
def __init__(self,
              wrapper: ArrayWrapper,
              close: tp.ArrayLike,
              order_records: tp.RecordArray,
              log_records: tp.RecordArray,
              init_cash: tp.ArrayLike,
              cash_sharing: bool,
              call_seq: tp.Optional[tp.Array2d] = None,
              fillna_close: tp.Optional[bool] = None,
              trades_type: tp.Optional[tp.Union[int, str]] = None) -> None:
    Wrapping.__init__(
        self,
        wrapper,
        close=close,
        order_records=order_records,
        log_records=log_records,
        init_cash=init_cash,
        cash_sharing=cash_sharing,
        call_seq=call_seq,
        fillna_close=fillna_close,
        trades_type=trades_type
    )
    StatsBuilderMixin.__init__(self)
    PlotsBuilderMixin.__init__(self)

    # Get defaults
    from vectorbt._settings import settings
    portfolio_cfg = settings['portfolio']

    if fillna_close is None:
        fillna_close = portfolio_cfg['fillna_close']
    if trades_type is None:
        trades_type = portfolio_cfg['trades_type']
    if isinstance(trades_type, str):
        trades_type = map_enum_fields(trades_type, TradesType)

    self._close = broadcast_to(close, wrapper.dummy(group_by=False))
    self._order_records = order_records
    self._log_records = log_records
    self._init_cash = init_cash
    self._cash_sharing = cash_sharing
    self._call_seq = call_seq
    self._fillna_close = fillna_close
    self._trades_type = trades_type
```

## indexing_func
根据索引操作 `pd_indexing_func` 选择出对应收盘价、订单记录、日志记录、初始现金、调用序列后，构建新的 `Portfolio` 对象返回。

## from_orders